# Causal Inference Exploration (DoWhy)
In this notebook, we interactively explore causal relationships. Traditional ML finds correlations, but we want to prove that an increase in Insider Buying structurally *causes* market outperformance, using Microsoft's DoWhy framework.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import dowhy
from dowhy import CausalModel
import warnings
warnings.filterwarnings('ignore')

### 1. Load the Processed Dataset

In [ ]:
repo_root = Path.cwd().parent
master_path = repo_root / "data" / "processed" / "master_dataset.parquet"
try:
    df = pd.read_parquet(master_path)
    df = df.dropna(subset=['target_score', 'insider_net_intensity', 'theme_increase'])
    print(f"Data loaded with shape: {df.shape}")
except Exception as e:
    print(f"Error loading data: {e}")

### 2. Define the Causal Model
We define our Treatment (Insider Intensity), Outcome (Target Score), and Common Causes/Confounders (Macro Regime, Volatility, Options Skew).

In [ ]:
model = CausalModel(
    data=df,
    treatment='insider_net_intensity',
    outcome='target_score',
    common_causes=['macro_regime', 'vol_63d', 'opt_put_call_vol_ratio_z']
)
model.view_model()
identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)
print(identified_estimand)

### 3. Estimate Average Treatment Effect (ATE)

In [ ]:
estimate = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.linear_regression",
    test_significance=True
)
print(f"Causal Estimate is {estimate.value}")

### 4. Robustness Check (Placebo Data Refutation)

In [ ]:
refute_results = model.refute_estimate(
    identified_estimand, estimate,
    method_name="random_common_cause"
)
print(refute_results)